In [2]:
import pandas as pd
import matplotlib.pyplot as plt
from linearmodels.panel import PanelOLS
import numpy as np

In [14]:
INDEX_COLS = ['alpha', 'beta', 'gamma']
QUARTER_PANEL_PATH = '../data/checkpoints/ready/panel_q.csv'
BIANNUAL_PANEL_PATH = '../data/checkpoints/ready/panel_h.csv'
FULL_PANEL_PATH = '../data/checkpoints/ready/panel_full.csv'

TELEWORK_PATH = '../data/telework_cps_level.csv'

GRAPHS_PATH = '../temp/graphics/graphs/'
TABLES_PATH = '../temp/graphics/tables/'
EVENT_STUDY_FOLDER = 'event_studies/'


REFERENCE_QUINTILE = 1

quintiles = [1,2,3,4,5]
quintiles.remove(REFERENCE_QUINTILE)

DV_COL = 'Y_SHARE'
DV_NAME = 'Entry-Level Workers Share'

In [28]:
panel_q = pd.read_csv(QUARTER_PANEL_PATH)
panel_h = pd.read_csv(BIANNUAL_PANEL_PATH)
panel_full = pd.read_csv(FULL_PANEL_PATH)

tele = pd.read_csv(TELEWORK_PATH)

In [5]:
print(panel_q.columns)

Index(['OCC', 'QUARTER', 'N', 'np', 'alpha', 'beta', 'gamma', 'Y_SHARE',
       'arcsinh_np', 'arcsinh_N', 'alpha_quintile', 'alpha_quintile_1',
       'alpha_quintile_2', 'alpha_quintile_3', 'alpha_quintile_4',
       'alpha_quintile_5', 'beta_quintile', 'beta_quintile_1',
       'beta_quintile_2', 'beta_quintile_3', 'beta_quintile_4',
       'beta_quintile_5', 'gamma_quintile', 'gamma_quintile_1',
       'gamma_quintile_2', 'gamma_quintile_3', 'gamma_quintile_4',
       'gamma_quintile_5'],
      dtype='str')


In [6]:
### Main DiD
# DV ~ POST:beta_quintile

for quintile in quintiles:
    df_did = panel_h.set_index(['OCC', 'HALF_YEAR'], drop=False)
    df_did = df_did[df_did['beta_quintile'].isin([REFERENCE_QUINTILE, quintile])]
    df_did["POST"] = df_did['HALF_YEAR'] >= 0

    did_formula = f"{DV_COL} ~ POST:beta_quintile_{quintile} + EntityEffects + TimeEffects"
    did_mod = PanelOLS.from_formula(did_formula, data=df_did, weights=df_did['N'])
    res_did = did_mod.fit(cov_type='clustered', cluster_entity=True)
    print(res_did.summary)


                          PanelOLS Estimation Summary                           
Dep. Variable:                Y_SHARE   R-squared:                        0.0036
Estimator:                   PanelOLS   R-squared (Between):             -0.0212
No. Observations:                1233   R-squared (Within):              -0.0094
Date:                Thu, Jul 30 2026   R-squared (Overall):             -0.0211
Time:                        19:46:04   Log-likelihood                    2899.4
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      3.9274
Entities:                         137   P-value                           0.0478
Avg Obs:                       9.0000   Distribution:                  F(1,1087)
Min Obs:                       9.0000                                           
Max Obs:                       9.0000   F-statistic (robust):             2.1335
                            

In [7]:
## Quarterly aggregation - since we did parallel trends at this level

for quintile in quintiles:
    df_did = panel_q.set_index(['OCC', 'QUARTER'], drop=False)
    df_did = df_did[df_did['beta_quintile'].isin([REFERENCE_QUINTILE, quintile])]
    df_did["POST"] = df_did['QUARTER'] >= 0

    did_formula = f"{DV_COL} ~ POST:beta_quintile_{quintile} + EntityEffects + TimeEffects"
    did_mod = PanelOLS.from_formula(did_formula, data=df_did, weights=df_did['N'])
    res_did = did_mod.fit(cov_type='clustered', cluster_entity=True)
    print(res_did.summary)


                          PanelOLS Estimation Summary                           
Dep. Variable:                Y_SHARE   R-squared:                        0.0024
Estimator:                   PanelOLS   R-squared (Between):             -0.0215
No. Observations:                2329   R-squared (Within):              -0.0062
Date:                Thu, Jul 30 2026   R-squared (Overall):             -0.0213
Time:                        19:46:04   Log-likelihood                    4904.3
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      5.2127
Entities:                         137   P-value                           0.0225
Avg Obs:                       17.000   Distribution:                  F(1,2175)
Min Obs:                       17.000                                           
Max Obs:                       17.000   F-statistic (robust):             2.4464
                            

In [8]:
## Eliminate the CPS redeign

for quintile in quintiles:
    df_did = panel_q.set_index(['OCC', 'QUARTER'], drop=False)
    df_did = df_did[df_did['beta_quintile'].isin([REFERENCE_QUINTILE, quintile])]
    
    # April 2025 lands in quarter 9
    df_did = df_did[df_did['QUARTER'] < 9]
    
    df_did["POST"] = df_did['QUARTER'] >= 0

    did_formula = f"{DV_COL} ~ POST:beta_quintile_{quintile} + EntityEffects + TimeEffects"
    did_mod = PanelOLS.from_formula(did_formula, data=df_did, weights=df_did['N'])
    res_did = did_mod.fit(cov_type='clustered', cluster_entity=True)
    print(res_did.summary)


                          PanelOLS Estimation Summary                           
Dep. Variable:                Y_SHARE   R-squared:                        0.0022
Estimator:                   PanelOLS   R-squared (Between):             -0.0173
No. Observations:                1781   R-squared (Within):              -0.0072
Date:                Thu, Jul 30 2026   R-squared (Overall):             -0.0171
Time:                        19:46:04   Log-likelihood                    3758.9
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      3.5905
Entities:                         137   P-value                           0.0583
Avg Obs:                       13.000   Distribution:                  F(1,1631)
Min Obs:                       13.000                                           
Max Obs:                       13.000   F-statistic (robust):             1.9073
                            

In [9]:
### Square Root Weights

for quintile in quintiles:
    df_did = panel_h.set_index(['OCC', 'HALF_YEAR'], drop=False)
    df_did = df_did[df_did['beta_quintile'].isin([REFERENCE_QUINTILE, quintile])]
    df_did["POST"] = df_did['HALF_YEAR'] >= 0
    
    df_did['sqrt_N'] = np.sqrt(df_did['N'])

    did_formula = f"{DV_COL} ~ POST:beta_quintile_{quintile} + EntityEffects + TimeEffects"
    did_mod = PanelOLS.from_formula(did_formula, data=df_did, weights=df_did['sqrt_N'])
    res_did = did_mod.fit(cov_type='clustered', cluster_entity=True)
    print(res_did.summary)


                          PanelOLS Estimation Summary                           
Dep. Variable:                Y_SHARE   R-squared:                        0.0027
Estimator:                   PanelOLS   R-squared (Between):             -0.0274
No. Observations:                1233   R-squared (Within):              -0.0059
Date:                Thu, Jul 30 2026   R-squared (Overall):             -0.0269
Time:                        19:46:04   Log-likelihood                    2487.2
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      2.9549
Entities:                         137   P-value                           0.0859
Avg Obs:                       9.0000   Distribution:                  F(1,1087)
Min Obs:                       9.0000                                           
Max Obs:                       9.0000   F-statistic (robust):             3.0267
                            

In [10]:
### No weights

for quintile in quintiles:
    df_did = panel_h.set_index(['OCC', 'HALF_YEAR'], drop=False)
    df_did = df_did[df_did['beta_quintile'].isin([REFERENCE_QUINTILE, quintile])]
    df_did["POST"] = df_did['HALF_YEAR'] >= 0
    
    df_did['sqrt_N'] = np.sqrt(df_did['N'])

    did_formula = f"{DV_COL} ~ POST:beta_quintile_{quintile} + EntityEffects + TimeEffects"
    did_mod = PanelOLS.from_formula(did_formula, data=df_did)
    res_did = did_mod.fit(cov_type='clustered', cluster_entity=True)
    print(res_did.summary)


                          PanelOLS Estimation Summary                           
Dep. Variable:                Y_SHARE   R-squared:                        0.0016
Estimator:                   PanelOLS   R-squared (Between):             -0.0306
No. Observations:                1233   R-squared (Within):              -0.0013
Date:                Thu, Jul 30 2026   R-squared (Overall):             -0.0292
Time:                        19:46:04   Log-likelihood                    2116.7
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      1.7928
Entities:                         137   P-value                           0.1809
Avg Obs:                       9.0000   Distribution:                  F(1,1087)
Min Obs:                       9.0000                                           
Max Obs:                       9.0000   F-statistic (robust):             1.4846
                            

In [ ]:
### Full Sample no weights

for quintile in quintiles:
    df_did = panel_full.set_index(['OCC', 'HALF_YEAR'], drop=False)
    df_did = df_did[df_did['beta_quintile'].isin([REFERENCE_QUINTILE, quintile])]
    df_did["POST"] = df_did['HALF_YEAR'] >= 0

    did_formula = f"{DV_COL} ~ POST:beta_quintile_{quintile} + EntityEffects + TimeEffects"
    did_mod = PanelOLS.from_formula(did_formula, data=df_did)
    res_did = did_mod.fit(cov_type='clustered', cluster_entity=True)
    print(res_did.summary)


                          PanelOLS Estimation Summary                           
Dep. Variable:                Y_SHARE   R-squared:                     4.809e-05
Estimator:                   PanelOLS   R-squared (Between):             -0.0073
No. Observations:                1853   R-squared (Within):              -0.0006
Date:                Thu, Jul 30 2026   R-squared (Overall):             -0.0066
Time:                        19:46:04   Log-likelihood                    2555.5
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      0.0788
Entities:                         206   P-value                           0.7790
Avg Obs:                       8.9951   Distribution:                  F(1,1638)
Min Obs:                       8.0000                                           
Max Obs:                       9.0000   F-statistic (robust):             0.0800
                            

In [ ]:
### Full Sample, but N>30 no weights, Around 200 per quintile

for quintile in quintiles:
    df_did = panel_full.set_index(['OCC', 'HALF_YEAR'], drop=False)
    df_did = df_did[df_did['beta_quintile'].isin([REFERENCE_QUINTILE, quintile])]
    df_did["POST"] = df_did['HALF_YEAR'] >= 0

    df_did = df_did[df_did["N"] >= 30]
    print(df_did['OCC'].nunique())

    did_formula = f"{DV_COL} ~ POST:beta_quintile_{quintile} + EntityEffects + TimeEffects"
    did_mod = PanelOLS.from_formula(did_formula, data=df_did)
    res_did = did_mod.fit(cov_type='clustered', cluster_entity=True)
    print(res_did.summary)

200
                          PanelOLS Estimation Summary                           
Dep. Variable:                Y_SHARE   R-squared:                        0.0004
Estimator:                   PanelOLS   R-squared (Between):             -0.0155
No. Observations:                1654   R-squared (Within):              -0.0026
Date:                Thu, Jul 30 2026   R-squared (Overall):             -0.0153
Time:                        19:47:26   Log-likelihood                    2685.0
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      0.5284
Entities:                         200   P-value                           0.4674
Avg Obs:                       8.2700   Distribution:                  F(1,1445)
Min Obs:                       1.0000                                           
Max Obs:                       9.0000   F-statistic (robust):             0.4305
                        

In [ ]:
# DV ~ POST:beta_quintile + POST:telework

### With a telework control

# Merge once, outside the loop — inner join replaces the left+indicator+filter dance
df_did_base = panel_h.merge(tele, left_on="OCC", right_on="CPS Code", how="inner", indicator=True)
print(df_did_base['OCC'].nunique())
# Down to 347 occupations
print(df_did_base['teleworkable'].describe())
df_did_base['tele_dummy'] = (df_did_base['teleworkable'] >= 0.5).astype(int)

for quintile in quintiles:
    # Fresh subset of the untouched base every iteration — nothing carries over
    df_q = df_did_base[df_did_base['beta_quintile'].isin([REFERENCE_QUINTILE, quintile])].copy()
    df_q = df_q.set_index(['OCC', 'HALF_YEAR'], drop=False)
    df_q["POST"] = df_q['HALF_YEAR'] >= 0

    did_formula = f"{DV_COL} ~ POST:beta_quintile_{quintile} + POST:tele_dummy + EntityEffects + TimeEffects"
    did_mod = PanelOLS.from_formula(did_formula, data=df_q, weights=df_q['N'])
    res_did = did_mod.fit(cov_type='clustered', cluster_entity=True)
    print(res_did.summary)

347
count    3123.000000
mean        0.379771
std         0.464619
min         0.000000
25%         0.000000
50%         0.000000
75%         1.000000
max         1.140857
Name: teleworkable, dtype: float64
                          PanelOLS Estimation Summary                           
Dep. Variable:                Y_SHARE   R-squared:                        0.0048
Estimator:                   PanelOLS   R-squared (Between):             -0.0200
No. Observations:                1233   R-squared (Within):              -0.0071
Date:                Thu, Jul 30 2026   R-squared (Overall):             -0.0199
Time:                        20:23:25   Log-likelihood                    2900.1
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      2.6357
Entities:                         137   P-value                           0.0721
Avg Obs:                       9.0000   Distribution:           

In [61]:
# telework as a sample split (not teleworkable)

df_did = df_did_base[df_did_base['tele_dummy'] == 0].copy()

print(f"Surviving N in this sample (Quintile 1): {df_did[df_did[f'beta_quintile_1'] == 1]['OCC'].nunique()}")
for quintile in quintiles:
    print(f"Surviving N in this sample (Quintile {quintile}): {df_did[df_did[f'beta_quintile_{quintile}'] == 1]['OCC'].nunique()}")

for quintile in quintiles:
    # Fresh subset of the untouched base every iteration — nothing carries over
    df_q = df_did[df_did['beta_quintile'].isin([REFERENCE_QUINTILE, quintile])].copy()
    df_q = df_q.set_index(['OCC', 'HALF_YEAR'], drop=False)
    df_q["POST"] = df_q['HALF_YEAR'] >= 0

    did_formula = f"{DV_COL} ~ POST:beta_quintile_{quintile} + EntityEffects + TimeEffects"
    did_mod = PanelOLS.from_formula(did_formula, data=df_q, weights=df_q['N'])
    res_did = did_mod.fit(cov_type='clustered', cluster_entity=True)
    print(res_did.summary)

Surviving N in this sample (Quintile 1): 68
Surviving N in this sample (Quintile 2): 57
Surviving N in this sample (Quintile 3): 47
Surviving N in this sample (Quintile 4): 24
Surviving N in this sample (Quintile 5): 17
                          PanelOLS Estimation Summary                           
Dep. Variable:                Y_SHARE   R-squared:                        0.0043
Estimator:                   PanelOLS   R-squared (Between):             -0.0210
No. Observations:                1125   R-squared (Within):              -0.0093
Date:                Thu, Jul 30 2026   R-squared (Overall):             -0.0209
Time:                        20:41:43   Log-likelihood                    2703.1
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      4.2864
Entities:                         125   P-value                           0.0387
Avg Obs:                       9.0000   Distributio

In [60]:
# telework as a sample split (teleworkable)

df_did = df_did_base[df_did_base['tele_dummy'] == 1].copy()

print(f"Surviving N in this sample (Quintile 1): {df_did[df_did[f'beta_quintile_1'] == 1]['OCC'].nunique()}")
for quintile in quintiles:
    print(f"Surviving N in this sample (Quintile {quintile}): {df_did[df_did[f'beta_quintile_{quintile}'] == 1]['OCC'].nunique()}")

for quintile in quintiles:
    # Fresh subset of the untouched base every iteration — nothing carries over
    df_q = df_did[df_did['beta_quintile'].isin([REFERENCE_QUINTILE, quintile])].copy()
    df_q = df_q.set_index(['OCC', 'HALF_YEAR'], drop=False)
    df_q["POST"] = df_q['HALF_YEAR'] >= 0

    did_formula = f"{DV_COL} ~ POST:beta_quintile_{quintile} + EntityEffects + TimeEffects"
    did_mod = PanelOLS.from_formula(did_formula, data=df_q, weights=df_q['N'])
    res_did = did_mod.fit(cov_type='clustered', cluster_entity=True)
    print(res_did.summary)

Surviving N in this sample (Quintile 1): 2
Surviving N in this sample (Quintile 2): 10
Surviving N in this sample (Quintile 3): 21
Surviving N in this sample (Quintile 4): 45
Surviving N in this sample (Quintile 5): 56
                          PanelOLS Estimation Summary                           
Dep. Variable:                Y_SHARE   R-squared:                        0.0047
Estimator:                   PanelOLS   R-squared (Between):             -0.0870
No. Observations:                 108   R-squared (Within):              -0.0417
Date:                Thu, Jul 30 2026   R-squared (Overall):             -0.0856
Time:                        20:41:24   Log-likelihood                    204.79
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      0.4110
Entities:                          12   P-value                           0.5231
Avg Obs:                       9.0000   Distribution